In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# 衝突判定のチェック
+ 機体点群を間引いた後のデータで、適当な人工データに対して問題なく衝突判定ができるか、衝突部位がおかしくないかなどを確認する

In [ ]:
import os
os.chdir("../")

## AppConfigの読み込み
+ 書き換えたいパラメータは書き換えられるように関数を読み込んでおく

In [ ]:
from configparser import ConfigParser, ExtendedInterpolation
from argus_synchro.experiments.config_replace import with_frozen_app_config
from argus_synchro.config.app_config import AppConfig

In [ ]:
app_ini = ConfigParser(interpolation=ExtendedInterpolation())
app_ini.read("./config/settings.ini", "UTF-8")
# 共有メモリに反映
app_config = AppConfig(app_ini)

## 関数準備

In [ ]:
from dataclasses import dataclass
from typing import Callable

import numpy as np
import open3d as o3d
import k3d

import argus_synchro.SubScrutinizer as SubScrt
import argus_synchro.point_processing.collision_indicator as col_ind
from argus_synchro.experiments.debug_vis.viewer_3d import (
    o3dlineset_to_k3dline,
    create_simple_k3d_line,
    create_simple_k3d_points,
    create_simple_k3d_mesh,
    o3dmesh_to_k3dmesh,
)

In [ ]:
col_sphere_radius = 0.075
col_cylinder_radius = 0.05

col_cylinder_color = [0.17, 0.7, 0.17]
col_sphere_color = [0, 0.5, 0]

In [ ]:
def np2pcd(
    data_array: np.ndarray,
    color_list: list = None,
) -> o3d.geometry.PointCloud:
    pcd = o3d.geometry.PointCloud()
    if len(data_array) == 0:
        return pcd

    pcd.points = o3d.utility.Vector3dVector(data_array)
    if color_list:
        pcd.paint_uniform_color(color_list)
    return pcd

In [ ]:
def concat_o3d_obj(o3d_objs: list[o3d.geometry.Geometry3D]) -> o3d.geometry.Geometry3D:
    if isinstance(o3d_objs[0], o3d.geometry.LineSet):
        concated = o3d.geometry.LineSet()
    elif isinstance(o3d_objs[0], o3d.geometry.PointCloud):
        concated = o3d.geometry.PointCloud()
    elif isinstance(o3d_objs[0], o3d.geometry.TriangleMesh):
        concated = o3d.geometry.TriangleMesh()
    else:
        raise ValueError(f"unexpected type of o3d_objs[0] = {type(o3d_objs[0])}")
    for o3d_obj in o3d_objs:
        concated += o3d_obj
    return concated

In [ ]:
def create_sphere(translate: tuple, radius: float = 0.1, color: tuple = (0, 0, 0)) -> o3d.geometry.TriangleMesh:
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius = radius)
    sphere.translate(translate)
    sphere.paint_uniform_color(color)
    return sphere


In [ ]:
def create_cylinder(point_from: tuple, point_to: tuple, radius: float = 0.05, color: tuple = (1, 0, 0)) -> o3d.geometry.TriangleMesh | None:
    np_point_from = np.array(point_from)
    np_point_to = np.array(point_to)
    
    direction = np_point_to - np_point_from
    center = (np_point_from + np_point_to) / 2
    length = np.linalg.norm(direction)

    if length == 0:
        return None

    cylinder = o3d.geometry.TriangleMesh.create_cylinder(radius = radius, height=length)

    rot_mat = col_ind.rotation_matrix_from_vectors(np.array([0, 0, 1]), direction)
    cylinder.rotate(rot_mat)
    cylinder.translate(center)
    return cylinder

In [ ]:
# def create_collision_obj(
#     collision_clusters: dict,
#     LS_pcd_det: LSC.LShared_pcd_det,
#     sphere_collision_color: list = [0, 0.5, 0],
#     shpere_radius: float = 0.05,
#     cylinder_radius: float = 0.025,
# ) -> (o3d.geometry.TriangleMesh, o3d.geometry.LineSet, o3d.geometry.TriangleMesh):
#     LS_pcd_det.collision_points, LS_pcd_det.collision_lines, LS_pcd_det.collision_colors = collision_indicator.create_line(obj_dict=collision_clusters)
    
#     # 衝突判定用円筒生成
#     LS_pcd_det.collision_vertices, LS_pcd_det.collision_triangles = collision_indicator.create_cylinder(obj_dict=collision_clusters, radius=cylinder_radius)
#     cylinder_collision = o3d.geometry.TriangleMesh(
#         vertices=LS_pcd_det.collision_vertices,
#         triangles=LS_pcd_det.collision_triangles,
#     )
    
#     line_collision = o3d.geometry.LineSet(
#         points=LS_pcd_det.collision_points,
#         lines=LS_pcd_det.collision_lines,
#     )
#     line_collision.colors = LS_pcd_det.collision_colors
    
#     sphere_collision = Subvisualize.create_spheres_at_line_endpoints(
#         line_collision,
#         radius=shpere_radius,
#         color=sphere_collision_color,
#     )
    
#     return (cylinder_collision, line_collision, sphere_collision)
#     pass

## 物体の設定

In [ ]:
def create_metal_pipe(
    pipe_radius: float = 0.048/2,
    pipe_width: float = 1.5,
    pipe_height: float = 0.5,
    pipe_width_offset: float = 0.05,
) -> o3d.geometry.TriangleMesh:
    # radius = 0.048/2
    # width = 1.5
    # height = 0.5
    # delta = 0.05
    pillar_offset = pipe_width/2-pipe_radius*2-pipe_width_offset
    
    single_pipe1 = o3d.geometry.TriangleMesh.create_cylinder(radius=pipe_radius, height=pipe_width)
    single_pipe2 = o3d.geometry.TriangleMesh.create_cylinder(radius=pipe_radius, height=pipe_height)
    single_pipe3 = o3d.geometry.TriangleMesh.create_cylinder(radius=pipe_radius, height=pipe_height)
    
    single_pipe1 = single_pipe1\
        .rotate(single_pipe1.get_rotation_matrix_from_axis_angle([0, np.pi/2, 0]))\
        .translate([0, 0, pipe_height])
    single_pipe2 = single_pipe2.translate([pillar_offset, 0, pipe_height/2])
    single_pipe3 = single_pipe3.translate([-pillar_offset, 0, pipe_height/2])
    
    single_pipe = single_pipe1 + single_pipe2 + single_pipe3
    return single_pipe

In [ ]:
def create_human(
    human_width: float = 0.5,
    human_depth: float = 0.5,
    human_height: float = 1.6,
    hand_height: float = 0.9,
    hand_width: float = 0.1,
    hand_length: float = 0.4,
    hand_depth: float = 0.5,
) -> o3d.geometry.TriangleMesh:
    # 球体を半分に切って、頭部を作成
    head = o3d.geometry.TriangleMesh.create_sphere(radius = human_width/2)
    rm_ind = (np.asarray(head.vertices)[:,2] < 0).tolist()
    head.remove_vertices_by_mask(rm_ind)
    head = head.translate([0, 0, human_height])

    # 直方体で体部分作成
    body = o3d.geometry.TriangleMesh.create_box(width=human_width, height=human_height, depth=human_depth)
    body = body.translate([-human_width/2, -human_height/2, -human_depth/2])
    body = body.rotate(body.get_rotation_matrix_from_axis_angle([np.pi/2, 0, 0]))
    body = body.translate([0, 0, human_height/2])
    
    # 直方体で手部分作成
    hand_r = o3d.geometry.TriangleMesh.create_box(width=hand_width, height=hand_depth, depth=hand_depth)
    hand_r = hand_r.translate([human_width/2, -hand_depth/2, -hand_depth/2 + hand_height])
    hand_l = o3d.geometry.TriangleMesh.create_box(width=hand_width, height=hand_length, depth=hand_depth)
    hand_l = hand_l.translate([-human_width/2-hand_width, -hand_length/2, -hand_depth/2 + hand_height])

    return head + body + hand_r + hand_l

In [ ]:
def create_cone(
    total_width: float = 0.38,
    total_depth: float = 0.38,
    total_height: float = 0.7,
    gnd_cone_height_offset: float = 0.05,
    gnd_cone_width_offset: float = 0.05,
) -> o3d.geometry.TriangleMesh:    
    # pylon_gnd_
    # pylon_gnd_width = 0.38,
    # pylon_gnd_depth = 0.38,
    # pylon_gnd_height = 0.05,
    # pylon_cone_offset = 0.05,
    # pylon_cone_height = 0.65,

    cone_radius = (total_width - gnd_cone_width_offset*2)/2
    cone_height = total_height - gnd_cone_height_offset
    
    simple_cone = o3d.geometry.TriangleMesh.create_cone(radius = cone_radius, height = cone_height)
    simple_cone = simple_cone.translate([0, 0, gnd_cone_height_offset])
    
    simple_box = o3d.geometry.TriangleMesh.create_box(width=total_width, depth=gnd_cone_height_offset, height=total_depth)
    simple_box = simple_box.translate([-total_width/2, -total_depth/2, 0])
    return simple_cone + simple_box

In [ ]:
def create_expanding_hand_human(
    human_width: float = 0.5,
    human_depth: float = 0.5,
    human_height: float = 1.6,
    hand_height: float = 0.9,
    hand_width: float = 0.1,
    hand_length: float = 0.4,
    hand_depth: float = 0.5,
    hand_radius: float = 0.1,
    hand_expand_length: float = 0.7,
) -> o3d.geometry.TriangleMesh:
    # 球体を半分に切って、頭部を作成
    head = o3d.geometry.TriangleMesh.create_sphere(radius = human_width/2)
    rm_ind = (np.asarray(head.vertices)[:,2] < 0).tolist()
    head.remove_vertices_by_mask(rm_ind)
    head = head.translate([0, 0, human_height])

    # 直方体で体部分作成
    body = o3d.geometry.TriangleMesh.create_box(width=human_width, height=human_height, depth=human_depth)
    body = body.translate([-human_width/2, -human_height/2, -human_depth/2])
    body = body.rotate(body.get_rotation_matrix_from_axis_angle([np.pi/2, 0, 0]))
    body = body.translate([0, 0, human_height/2])
    
    # 直方体で手部分作成
    hand_r = o3d.geometry.TriangleMesh.create_box(width=hand_width, height=hand_depth, depth=hand_depth)
    hand_r = hand_r.translate([human_width/2, -hand_depth/2, -hand_depth/2 + hand_height])   
    hand_l = o3d.geometry.TriangleMesh.create_box(width=hand_width, height=hand_length, depth=hand_depth)
    hand_l = hand_l.translate([-human_width/2-hand_width, -hand_length/2, -hand_depth/2 + hand_height])
    
    # 手を伸ばした部分を作成
    hand_expand_r = o3d.geometry.TriangleMesh.create_cylinder(radius=hand_radius, height=hand_expand_length)
    hand_expand_r = hand_expand_r\
    .rotate(hand_expand_r.get_rotation_matrix_from_axis_angle([0, np.pi/2, 0]))\
    .translate([human_width + hand_width, 0, hand_height])
    
    hand_expand_l = o3d.geometry.TriangleMesh.create_cylinder(radius=hand_radius, height=hand_expand_length)
    hand_expand_l = hand_expand_l\
    .rotate(hand_expand_l.get_rotation_matrix_from_axis_angle([0, np.pi/2, 0]))\
    .translate([-human_width - hand_width, 0, hand_height])
    
    return head + body + hand_r + hand_l + hand_expand_r + hand_expand_l

In [ ]:
def create_boxes(
    start_y:float,
    stop_y:float,
    n_boxes:int,
    width:float=1.0,
    height:float=1.0,
    depth:float=1.0,
    x_offset:float=0,
    z_offset:float=0,
):
    boxes = o3d.geometry.TriangleMesh()
    for y_pos in np.linspace(start=start_y, stop=stop_y, num=n_boxes):
        boxes += o3d.geometry.TriangleMesh.create_box(width=width, height=height, depth=depth).translate([x_offset, y_pos, z_offset])
        pass
    return boxes

In [ ]:
def _reverse_pos(reverses: list, ndarray: np.ndarray) -> np.ndarray:
    reverse_ndarray = ndarray @ np.diag(np.array([-1 if reverse else 1 for reverse in reverses])).T

    if not ini.getboolean("calibration", "calib_lidar2crane"):
        reverse_ndarray[:,1] += -2
        
    return reverse_ndarray

def create_lr_lidar_points(
    np_target_obj: np.ndarray,
    trans_mat: np.ndarray,
    inv_trans_mat: np.ndarray | None = None,
    noise_scale: float = 0.2,
    n_sample_lr: tuple = (0.5, 0.5),
    noise_trans_vec_r: tuple = (0, 0, 0),
    noise_rot_r: tuple = (0, 0, 0),
    noise_trans_vec_l: tuple = (0, 0, 0),
    noise_rot_l: tuple = (0, 0, 0),
) -> dict:
    n_sample = len(np_target_obj)
    n_sample_lr = (int(n_sample * n_sample_lr[0]), int(n_sample * n_sample_lr[1]))
    
    if inv_trans_mat is None:
        _inv_trans_mat = np.linalg.inv(trans_mat)
    else:
        _inv_trans_mat = inv_trans_mat

    # LiDARは元々の座標系で反対に付いているので反転させる
    np_target_obj = _reverse_pos([False, True, True], np_target_obj)

    # 右/左LiDARの座標系に物体を変換する
    quasi_lidar_obj_r = np_target_obj @ _inv_trans_mat[:3,:3].T + _inv_trans_mat[:3,-1]
    quasi_lidar_obj_l = np_target_obj

    # 疑似LiDAR点群上でノイズやズレを付与する
    quasi_lidar_r = quasi_lidar_obj_r[np.random.choice(np.arange(len(quasi_lidar_obj_r)), size=n_sample_lr[LID.RIGHT])]
    quasi_lidar_r += np.random.normal(scale = noise_scale, size = quasi_lidar_r.shape)
    quasi_lidar_r = quasi_lidar_r @ o3d.geometry.PointCloud.get_rotation_matrix_from_xyz(noise_rot_r).T + np.array(noise_trans_vec_r)

    quasi_lidar_l = quasi_lidar_obj_l[np.random.choice(np.arange(len(quasi_lidar_obj_l)), size=n_sample_lr[LID.LEFT])]
    quasi_lidar_l += np.random.normal(scale = noise_scale, size = quasi_lidar_l.shape)
    quasi_lidar_l = quasi_lidar_l @ o3d.geometry.PointCloud.get_rotation_matrix_from_xyz(noise_rot_l).T + np.array(noise_trans_vec_l)

    # 疑似LiDAR点群を後続処理で動かすようにデータを成形する
    quasi_lidar = {}
    quasi_lidar[LID.RIGHT] = quasi_lidar_r
    quasi_lidar[LID.LEFT] = quasi_lidar_l
    return quasi_lidar

# 軌跡の設定

## 長方形
+ 幅高さ(a,b)の長方形を作る
+ 原点から$\pm$+-(a/2, b/2)離れている点を4隅として持つ点を作る

In [ ]:
def create_trajectory_rectangle(
    rect_width: float = 9.0,
    rect_height: float = 7.0,
    z_height: float = -1.0,
    point_interval: float = 2.0,
) -> (list[np.ndarray], list[np.ndarray]):
    """ 幅rect_width, 奥行rect_height, 高さz_height上にある長方形の上にpoint_intervalの間隔で点を置いていく関数
    置いた点の位置と置いた点における回転角のリストを返す
    """
    rect_total_len = 2*rect_width + 2*rect_height
    delta_x = rect_width/2
    delta_y = rect_height/2
    
    param_t = np.arange(rect_total_len, step=point_interval)
    poses = []
    rotates = []
    for val in param_t % rect_total_len:
        if 0 <= val and val < rect_width:
            origin = np.array([-delta_x, delta_y, z_height])
            time = (val - 0)
            velocity = np.array([1, 0, 0])
            rotate = np.array([0, 0, np.pi/2])
            pass
        elif rect_width <= val and val < (rect_width + rect_height):
            origin = np.array([delta_x, delta_y, z_height])
            time = (val - rect_width)
            velocity = np.array([0, -1, 0])
            rotate = np.array([0, 0, 0])
            pass
        elif (rect_width + rect_height) <= val and val < (2*rect_width + rect_height):
            origin = np.array([delta_x, -delta_y, z_height])
            time = (val - (rect_width + rect_height))
            velocity = np.array([-1, 0, 0])
            rotate = np.array([0, 0, -np.pi/2])
            pass
        elif (2*rect_width + rect_height) <= val and val < rect_total_len:
            origin = np.array([-delta_x, -delta_y, z_height])
            time = (val - (2*rect_width + rect_height))
            velocity = np.array([0, 1, 0])
            rotate = np.array([0, 0, -np.pi])
            pass
        else:
            raise ValueError("val should be 0 <= val <= 2*rect_width+2*rect_height")
        poses.append(origin + time * velocity)
        rotates.append(rotate)
    return poses, rotates

In [ ]:
def create_rectangle_lineset(
    center: np.ndarray,
    rect_width: float = 9.0,
    rect_height: float = 7.0,
    z_height: float = -1.0,
    color_list: list = [0, 0, 0],
) -> o3d.geometry.LineSet:
    """ centerを中心として幅rect_width, 奥行rect_height, 高さz_heightの長方形を作る
    """
    delta_x = rect_width/2
    delta_y = rect_height/2
    rect_pos = [
        (delta_x, delta_y, z_height),
        (delta_x, -delta_y, z_height),
        (-delta_x, -delta_y, z_height),
        (-delta_x, delta_y, z_height),
    ]

    o3d_rect = o3d.geometry.LineSet()
    o3d_rect.points = o3d.utility.Vector3dVector(np.array(rect_pos) + center)
    o3d_rect.lines = o3d.utility.Vector2iVector(
        np.hstack([
            np.arange(0, 3),
            np.arange(1, 4),
            np.array([3, 0])
        ]).reshape((-1, 2))
    )
    o3d_rect.paint_uniform_color(color_list)
    return o3d_rect

In [ ]:
def create_rect_traj_around_machine(
    machine_points: np.ndarray,
    create_obj_callback: Callable,
    rect_width: float = 9.0,
    rect_height: float = 7.0,
    z_height: float = -1.0,
    point_interval: float = 2.0,
    color_list: list = [0, 0, 0],
    **obj_params,
) -> (o3d.geometry.LineSet, list, list, list):
    machine_center = (machine_points.max(axis=0) + machine_points.min(axis=0)) / 2
    machine_center[2] = 0

    o3d_rect = create_rectangle_lineset(machine_center, rect_width, rect_height, z_height, color_list)

    poses, rotates = create_trajectory_rectangle(
        rect_width,
        rect_height,
        z_height,
        point_interval,
    )

    created_objs = [
        create_obj_callback(**obj_params)
        .rotate(o3d.geometry.TriangleMesh.get_rotation_matrix_from_xyz(rotate))
        .translate(pos + machine_center)
        for pos, rotate in zip(poses, rotates)
    ]

    return o3d_rect, created_objs, poses, rotates, machine_center

## 機体の下

In [ ]:
@dataclass
class RectTrajectory:
    width: float
    height: float
    translate: tuple
    z: float

In [ ]:
rect_traj_under_driver = [
    RectTrajectory(
        2.5,
        0.4,
        (-3.0, 1.2, 0),
        -1.0,
    ),
    RectTrajectory(
        2.5,
        0.4,
        (-3.0, -1.0, 0),
        -1.0,
    ),    
    RectTrajectory(
        0.4,
        2.6,
        (-3.0, 0.1, 0),
        -1.0,
    ),
]

rect_traj_under_cw = [
    RectTrajectory(
        2.5,
        0.4,
        (3.0, 1.2, 0),
        -1.0,
    ),
    RectTrajectory(
        2.5,
        0.4,
        (3.0, -1.0, 0),
        -1.0,
    ),    
    RectTrajectory(
        0.4,
        2.6,
        (3.0, 0.1, 0),
        -1.0,
    ),
]


In [ ]:
def create_rect_traj_under_machine(
    rect_trajs: list[RectTrajectory],
    create_obj_callback: Callable,
    num_points: int = 10,
    rect_color: list = [1, 0, 0],
) -> tuple[list, list, list, list]:
    traj_linesets = []
    put_objs = []
    put_poses = []
    translate_id = [] # 点の配置箇所が被っていると、後処理で重複排除されてしまうので、translateの情報を追加
    
    for rect_traj in rect_trajs:
        traj_linesets.append(
            create_rectangle_lineset(
                center=np.array(rect_traj.translate),
                rect_width=rect_traj.width,
                rect_height=rect_traj.height,
                z_height=rect_traj.z,
                color_list=rect_color,
            )
        )
        
        poses, rotates = create_trajectory_rectangle(
            rect_traj.width,
            rect_traj.height,
            rect_traj.z,
            2*(rect_traj.width + rect_traj.height)/ num_points,
        )
    
        put_objs.extend([
            create_obj_callback()
            .rotate(o3d.geometry.TriangleMesh.get_rotation_matrix_from_xyz(rotate))
            .translate(pos + np.array(rect_traj.translate))
            for pos, rotate in zip(poses, rotates)        
        ])

        put_poses.extend(poses)
        translate_id.extend([rect_traj.translate] * len(poses))

    return traj_linesets, put_objs, put_poses, translate_id

In [ ]:
# num_points = 5
# create_little_human = lambda: create_human(human_height = 0.8, hand_height = 0.4)
# machine_points = np.vstack([machine_mobile_points_measure, machine_immobile_points_measure])

# trajs, created_objs, poses, translate_ids = create_rect_traj_under_machine(
#     rect_traj_under_cw + rect_traj_under_driver,
#     create_little_human,
#     # create_metal_pipe,
#     num_points,
# )

# plot = k3d.plot()
# plot += create_simple_k3d_line(*o3dlineset_to_k3dline(concat_o3d_obj(trajs)), color=0x0000ff)
# # plot += create_simple_k3d_points(np.array(poses) + machine_center, color=0x000000, point_size=0.1)

# plot += create_simple_k3d_points(
#     machine_points, 
#     color=0x000000,
#     point_size=0.1,
# )

# plot += create_simple_k3d_mesh(*o3dmesh_to_k3dmesh(concat_o3d_obj(created_objs)), color=0x0000ff)
# plot.display()

# 衝突部位のチェック
+ とりあえず、機体周辺の長方形に物体を配置して、変な場所に衝突部位が出ないか確認

## 機体点群の読み込み

In [ ]:
l_machine_col, machine_mobile_points_measure, machine_immobile_points_measure = SubScrt.create_machine_points(
    machine_dir=app_config.OctoTree.col_machine_dir, 
    lidarposition=app_config.LiDARPosition,
    json_file=app_config.OctoTree.json_col_machine_file,
)

In [ ]:
machine_points = machine_mobile_points_measure
# machine_center = (machine_points.max(axis=0) + machine_points.min(axis=0)) / 2
# machine_center[2] = 0

len_x, len_y, len_z = machine_points.max(axis=0) - machine_points.min(axis=0)

In [ ]:
additional_width = 0.5
additional_height = 0.5

## 長方形上の軌跡に点群を設置

In [ ]:
rect_width = len_x + additional_width
rect_height = len_y + additional_height
z_height = machine_points[:,2].min()
point_interval = 0.5
create_obj_callback = create_human

In [ ]:
o3d_rect, created_objs, poses, rotates, machine_center = create_rect_traj_around_machine(
    machine_points,
    create_obj_callback,
    rect_width,
    rect_height,
    z_height,
    point_interval,
)

In [ ]:
plot = k3d.plot()
plot += create_simple_k3d_line(*o3dlineset_to_k3dline(o3d_rect), color=0x0000ff)
plot += create_simple_k3d_points(np.array(poses) + machine_center, color=0x000000, point_size=0.1)

plot += create_simple_k3d_points(
    np.vstack([machine_immobile_points_measure, machine_mobile_points_measure]), 
    color=0x000000,
    point_size=0.05
)

plot += create_simple_k3d_mesh(*o3dmesh_to_k3dmesh(concat_o3d_obj(created_objs)), color=0x0000ff)
plot.display()

## 軌跡の各点での物体の衝突判定チェック

In [ ]:
from itertools import islice

from octotree import controller as octo_ctrl
from octotree.octotree import NodeEntity

from argus_synchro import SubScrutinizer as Sub
from argus_synchro.experiments.config_replace import with_frozen_app_config
from argus_synchro.interface.octotree_func import OctoTreeFuncOn
from argus_synchro.py_octotree.detectable_points import get_detectable_z_range
from argus_synchro import detect3d, SceneDesc

## 初期化

### 物体に対して必ず接触可能性ありと判定されるように設定する

In [ ]:
app_config.CollisionDetection = with_frozen_app_config(
    app_config.CollisionDetection,
    distance_threshold=None,
    dialate_point_size=app_config.OctoTree.max_tree_depth,
    detect_focus_range=[],
)

## 接触可能性探索の点群を用意

In [ ]:
det_point_mobile_gen, det_point_immobile_gen = SubScrt.initialize_detectable_point_generators(
    machine_mobile_points=machine_mobile_points_measure,
    machine_immobile_points=machine_immobile_points_measure,
    detectable_tree_depth=app_config.OctoTree.max_tree_depth - app_config.CollisionDetection.dialate_point_size,
    z_range=get_detectable_z_range(
        app_config.General,
        app_config.CollisionDetection,
    ),
    max_dist=app_config.CollisionDetection.max_dist,
    grid_intervals=app_config.CollisionDetection.grid_intervals,
    min_radius=app_config.CollisionDetection.min_radius,
    max_radius=app_config.CollisionDetection.max_radius,
    key_num=app_config.CollisionDetection.key_num,
    octotree_conf=app_config.OctoTree,
    dialate_point_size=app_config.CollisionDetection.dialate_point_size,
    offset_rotate_center=app_config.machine.offset_rotate_center,
)

## 八分木インスタンスの初期化

In [ ]:
octree_obj = SubScrt.initialize_octotree(
    machine_immobile_points_measure=machine_immobile_points_measure,
    machine_mobile_points_measure=machine_mobile_points_measure,
    max_xyz=app_config.OctoTree.max_xyz,
    min_xyz=app_config.OctoTree.min_xyz,
    max_tree_depth=app_config.OctoTree.max_tree_depth,
    use_node_stats=app_config.OctoTree.use_node_stats,
    dialate_point_size=app_config.CollisionDetection.dialate_point_size,
    origin_w2oct=(0.0, 0.0, 0.0)
)

# 機体点群の非可動部を八分木に入れる
octree_obj = SubScrt.put_immobile_points_to_octotree(
    octotree_obj=octree_obj,
    machine_immobile_points_measure=machine_immobile_points_measure,
    machine_immobile_points_detect=det_point_immobile_gen.get_detectable_points(yaw_angle=None),
    machine_center=app_config.machine.offset_rotate_center,
)

## その他の設定を追加

In [ ]:
app_config.CollisionDetection.coord_method

In [ ]:
scene = SceneDesc.Scene(app_config.SceneDescription)
octotree_func = OctoTreeFuncOn()
collision_detect_creator = SubScrt.initialize_collision_detector(
    app_config.CollisionDetection.func_on,
    app_config.CollisionDetection.collision_detector_name,
    "VOX_MED",    
)

## ループ処理
1. 八分木の更新がある属性の削除
2. 人工的に配置した点群をNDArrayに変換
3. 機体点群除去
4. 除去後の点群を八分木に格納
5. 格納したデータでクラスタリング
6. クラスタリング結果を八分木に格納
7. 機体の可動部を八分木に格納
8. 衝突判定
9. 結果を各変数に格納

In [ ]:
history_octree_lidar = dict()
history_octree_col_pos = dict()

# for translate_id, pos, created_obj in zip(translate_ids, poses, created_objs):
for pos, created_obj in zip(poses, created_objs):
    # 1. 八分木の更新がある属性の削除
    octree_obj.erase_nodes_for_entities_noret(
        [
            NodeEntity.UNK,
            NodeEntity.OTHER,
            NodeEntity.CLIFF,
            NodeEntity.CRANE_MOBILE,
            NodeEntity.CRANE_MOBILE_FOR_DET,
            NodeEntity.HUMAN,
            NodeEntity.LOW_3D,
        ]
    )

    # 2. 人工的に配置した点群をNDArrayに変換
    yaw_angle = 0
    o3d_target_obj = created_obj.sample_points_uniformly(1000)
    np_target_obj = np.array(o3d_target_obj.points)
    
    # 3. 機体点群除去
    rd1, rd2, rd3 = map(
        float,
        octree_obj.cell_interval
        * app_config.OctoTree.remove_dist,
    )
    np_target_obj = (
        octo_ctrl.remove_machine_points(
            pcd_points=np_target_obj,
            l_machine_col=l_machine_col,
            remove_dist=(rd1, rd2, rd3),
            yaw_angle=yaw_angle,
        )
    )

    # 4. 除去後の点群を八分木に格納
    downsampled_accum_points, octree_obj = (
        octotree_func.octotree_accum(
            np_target_obj,
            octree_obj,
            NodeEntity.OTHER,
            app_config.OctoTree.clustering_tree_depth,
        )
    )

    # 5. 格納したデータでクラスタリング
    boxes, _, minmax, valid_detects, labels = detect3d.main_accum(
        downsampled_accum_points,
        app_config.DEFAULT.debug_log,
        app_config.detect3d.eps,
        app_config.detect3d.min_samples,
    )
    clustering_labels = labels[: len(downsampled_accum_points)]

    # 6. クラスタリング結果を八分木に格納
    octotree_func.clustering_result(
        octree_obj,
        clustered_data=downsampled_accum_points,
        labels=clustering_labels[: len(downsampled_accum_points)],
        start_time=0,
        cluster_entity=NodeEntity.OTHER,
        cluster_fail_table={-1: NodeEntity.UNK},
    )

    # 7. 機体の可動部を八分木に格納
    octotree_func.update_machine_mobile(
        machine_mobile_points_measure=machine_mobile_points_measure,
        machine_mobile_points_detect=det_point_mobile_gen.get_detectable_points(yaw_angle=yaw_angle),
        octotree_obj=octree_obj,
        yaw_angle=yaw_angle,
    )

    # 8. 衝突判定
    collision_clusters = (
        collision_detect_creator.collision_detection(
            octotree_obj=octree_obj,
            app_config=app_config,
        )
    )
    collision_clusters = scene.append_distance_info(
        collision_clusters,
        minmax,
    )
    
    _, _, col_machine_pos, col_pcd_pos, _, _ = list(islice(collision_clusters.values(), 1))[0]
    
    # 9. 結果を各変数に格納
    # store_key = (translate_id, tuple(pos))
    store_key = tuple(pos)
    history_octree_lidar[store_key] = downsampled_accum_points
    history_octree_col_pos[store_key] = (col_machine_pos, col_pcd_pos)

# 結果の可視化

In [ ]:
from argus_synchro.experiments.debug_vis.viewer_3d import o3dpcd_to_k3dpoints

In [ ]:
col_cylinder = o3d.geometry.TriangleMesh()
col_sphere = o3d.geometry.TriangleMesh()

for col_machine_pos, col_pcd_pos in history_octree_col_pos.values():
    _cylinder = create_cylinder(col_machine_pos, col_pcd_pos, radius = col_cylinder_radius/2, color=col_cylinder_color)
    if _cylinder:
        col_cylinder += _cylinder
    col_sphere += create_sphere(col_machine_pos, radius=col_sphere_radius, color=col_sphere_color) + create_sphere(col_pcd_pos, radius=col_sphere_radius, color=col_sphere_color)

In [ ]:
pcd = concat_o3d_obj([np2pcd(pcd) for pcd in history_octree_lidar.values()])

In [ ]:
plot = k3d.plot()

plot += create_simple_k3d_mesh(
    *o3dmesh_to_k3dmesh(col_cylinder + col_sphere), 
    color=int('0x%02x%02x%02x' % tuple(map(lambda elem: int(255*elem) , col_cylinder_color)), 16),
)

plot += create_simple_k3d_points(o3dpcd_to_k3dpoints(pcd), point_size=0.05)
plot += create_simple_k3d_points(np.vstack([machine_immobile_points_measure, machine_mobile_points_measure]), color=0x00ff00, point_size=0.05)

plot.display()

# Todo
+ 最短部位の位置が変: VOX_MEDだと上手くいくが、統計量計算すると上手くいかない => vox_statsの一部がnullになっているので、コードの修正が必要

In [ ]:
from collections import ChainMap

from octotree.octotree import NodeClusterKey, NodeEntity

In [ ]:
pcd_octonodes = octree_obj.entity_octonodes[NodeClusterKey(NodeEntity.OTHER, 0)]

In [ ]:
{
    k: v.get_mean() # get_meanが統計量の平均値を返すが一部Noneになっていることが分かる
    for k, v in pcd_octonodes.items()
}